# How chrF and COMET are calculated

Using real model predictions from `test_predictions.json`.

In [ ]:
import json
from pathlib import Path

path = Path(r"D:\J\Desktop\language_technology\course\projects_AI\mt_oil_no\outputs\exp1_data_scaling\size_13935\seed_42\test_predictions.json")

with open(path, encoding="utf-8") as f:
    data = json.load(f)

sources     = [d["source"]     for d in data]
references  = [d["reference"]  for d in data]
predictions = [d["prediction"] for d in data]

print(f"{len(data)} samples")
print("source    :", sources[0])
print("reference :", references[0])
print("prediction:", predictions[0])

---
# Part 1 — chrF

chrF counts **character** n-gram overlap instead of word n-gram overlap.
The key difference from BLEU: it works at the character level, so partial word matches count.

This matters for Norwegian — `ventilen` and `ventil` share most characters,
BLEU sees them as completely different words, chrF gives partial credit.

## Step 1 — character n-grams

In [ ]:
pred = predictions[0]
ref  = references[0]

print("prediction:", pred)
print("reference :", ref)

In [ ]:
from collections import Counter

def char_ngrams(text, n):
    # remove spaces before extracting character n-grams
    text = text.replace(" ", "")
    return [text[i:i+n] for i in range(len(text)-n+1)]

pred_chars = char_ngrams(pred, 6)   # chrF uses 6-grams by default
ref_chars  = char_ngrams(ref,  6)

print("first 10 character 6-grams from prediction:")
print(pred_chars[:10])
print()
print("first 10 character 6-grams from reference:")
print(ref_chars[:10])

## Step 2 — precision and recall

Unlike BLEU which only measures precision, chrF computes both precision and recall,
then combines them into an F-score.

In [ ]:
pred_counts = Counter(pred_chars)
ref_counts  = Counter(ref_chars)

matched = sum(min(c, ref_counts[g]) for g, c in pred_counts.items())

precision = matched / len(pred_chars) if pred_chars else 0
recall    = matched / len(ref_chars)  if ref_chars  else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"matched char 6-grams : {matched}")
print(f"precision            : {matched}/{len(pred_chars)} = {precision:.4f}")
print(f"recall               : {matched}/{len(ref_chars)}  = {recall:.4f}")
print(f"chrF (F1)            : {f1:.4f}")

## Step 3 — verify against the library

In [ ]:
import evaluate

chrf = evaluate.load("chrf")
result = chrf.compute(predictions=predictions, references=references)

print("corpus chrF:", round(result["score"], 4))

## Step 4 — BLEU vs chrF on a morphological variant

This is why chrF matters for Norwegian.

In [ ]:
import evaluate
bleu = evaluate.load("bleu")

# 'konglomeratene' (prediction) vs 'konglomeratene av perm-trias alder' (reference)
# one word difference but they share most characters
p = [predictions[0]]
r = [references[0]]

b = bleu.compute(predictions=p, references=[r])
c = chrf.compute(predictions=p, references=r)

print("pred :", p[0])
print("ref  :", r[0])
print()
print("BLEU :", round(b["bleu"], 4))
print("chrF :", round(c["score"], 4))

---
# Part 2 — COMET

BLEU and chrF are string-matching metrics — they don't understand meaning.

COMET uses a neural model trained on human judgements of translation quality.
It takes **source + prediction + reference** and outputs a score between 0 and 1.

Key difference: COMET can give a high score even if the wording differs,
as long as the meaning is preserved.

## Step 1 — load the model

In [ ]:
from comet import download_model, load_from_checkpoint

comet_model = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))
print("model loaded")

## Step 2 — what the input looks like

In [ ]:
# COMET needs src + mt + ref for every sentence
# this is why it's the only metric that uses the source
comet_data = [
    {"src": src, "mt": pred, "ref": ref}
    for src, pred, ref in zip(sources, predictions, references)
]

print("example input to COMET:")
print(json.dumps(comet_data[0], indent=2, ensure_ascii=False))

## Step 3 — run COMET on a small batch first

In [ ]:
result_small = comet_model.predict(comet_data[:5], batch_size=4, accelerator="auto")

for i, (item, score) in enumerate(zip(comet_data[:5], result_small["scores"])):
    print(f"[{i}] score: {score:.4f}")
    print(f"     src : {item['src']}")
    print(f"     pred: {item['mt']}")
    print(f"     ref : {item['ref']}")
    print()

## Step 4 — run on the full test set

In [ ]:
import numpy as np

result = comet_model.predict(comet_data, batch_size=4, accelerator="auto")

print("system score (mean) :", round(result["system_score"], 4))
print("std across sentences:", round(float(np.std(result["scores"])), 4))

## Step 5 — find best and worst sentences by COMET score

In [ ]:
scores = result["scores"]

best_idx  = max(range(len(scores)), key=lambda i: scores[i])
worst_idx = min(range(len(scores)), key=lambda i: scores[i])

print(f"BEST  (COMET={scores[best_idx]:.4f}):")
print("  src :", sources[best_idx])
print("  pred:", predictions[best_idx])
print("  ref :", references[best_idx])
print()
print(f"WORST (COMET={scores[worst_idx]:.4f}):")
print("  src :", sources[worst_idx])
print("  pred:", predictions[worst_idx])
print("  ref :", references[worst_idx])

## Step 6 — compare all three metrics on the same predictions

In [ ]:
bleu_result = bleu.compute(predictions=predictions, references=[[r] for r in references])
chrf_result = chrf.compute(predictions=predictions, references=references)

print("BLEU :", round(bleu_result["bleu"], 4))
print("chrF :", round(chrf_result["score"], 4))
print("COMET:", round(result["system_score"], 4))